# Multi-modal Model V7 - Polar Coordinate Regression (H96)

이 노트북은 기존 H96 모델의 좌표 직접 회귀 방식을 **Polar Coordinate (거리 + 방향) 분해** 방식으로 변경합니다.

## 주요 변경 사항

1.  **Polar Coordinate Target**:
    - 기존: `(end_x, end_y)` 직접 예측
    - 변경: `(distance, sin_theta, cos_theta)` 예측 후 좌표 복원
2.  **Model Output**:
    - Distance Head: Softplus 활성화 (거리는 양수)
    - Direction Head: Tanh 활성화 (sin/cos는 [-1, 1])
3.  **Loss Function**:
    - Distance: Huber Loss (robust to outliers)
    - Direction: Cosine Similarity Loss
4.  **No Data Leakage**:
    - Polar 라벨은 train에서만 생성
    - Test에서는 start 좌표만 사용

## 왜 성능이 개선될 수 있는가?

- **물리적 해석**: 패스는 본질적으로 "얼마나 멀리, 어느 방향"으로 표현하는 것이 자연스러움
- **각도 불연속 제거**: (sin θ, cos θ) 사용으로 -π/π 경계 문제 해결
- **독립적 Loss**: 거리와 방향에 각각 최적화된 loss 적용


In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from sklearn.model_selection import GroupKFold
import cv2  # OpenCV Import
from tqdm import tqdm
import joblib

# [Polar] Use Polar Preprocessor
from src.preprocessing_polar import FootballPreprocessorPolar

# Fold-specific Seeds for Diversity
FOLD_SEEDS = {
    1: 42,
    2: 43,
    3: 44,
    4: 45,
    5: 46,
    6: 47,
    7: 48,
    8: 49,
    9: 50,
    10: 51,
}

# Set Seed
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


Device: cuda


## 1. 데이터 로드 및 전처리 (Preprocessing V2.5)

## 2. 이미지 생성 및 증강 (Random Y-Flip)

In [2]:
# Load Raw Training Data
train_df = pd.read_csv("train.csv")
print(f"Loaded {len(train_df)} rows")

Loaded 356721 rows


## 3. 모델 아키텍처 (GRU 적용)

In [3]:
# [FIXED] Dataset Class Definition
class MultiModalDataset(Dataset):
    def __init__(self, episodes, img_size=(68, 105), augment=False, cache_images=True):
        self.episodes = episodes
        self.H, self.W = img_size
        self.augment = augment
        self.cache_images = cache_images
        self.img_cache = None
        
        if self.cache_images:
            print(f"Pre-rendering {len(episodes)} images (Cache Enabled)...")
            self.img_cache = [self._generate_image(ep['cont']) for ep in tqdm(episodes, desc="Caching Images")]

    def __len__(self):
        return len(self.episodes)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
             return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val
        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        data = self.episodes[idx]
        
        cont = data['cont'].copy()
        target = data['target'].copy()
        target_polar = data['target_polar'].copy() # [d_norm, sin, cos]
        
        if self.augment and random.random() < 0.5:
            cont[:, 1] = 1.0 - cont[:, 1] # start_y
            cont[:, 3] = 1.0 - cont[:, 3] # end_y_prev
            cont[:, 5] = -cont[:, 5]      # dy_prev 
            target[1] = 1.0 - target[1]
            target_polar[1] = -target_polar[1] # sin -> -sin
            
            if self.cache_images:
                base_img = self.img_cache[idx]
                img = torch.flip(base_img, [1])
            else:
                img = self._generate_image(cont)
        else:
            if self.cache_images:
                img = self.img_cache[idx]
            else:
                img = self._generate_image(cont)
        
        cont_tensor = torch.tensor(cont, dtype=torch.float32)
        cat_tensor = torch.tensor(data['cat'], dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.float32)
        target_polar_tensor = torch.tensor(target_polar, dtype=torch.float32)
        
        seq_len = cont.shape[0]
        aux_target = np.zeros((seq_len, 2), dtype=np.float32)
        if seq_len > 1:
            aux_target[:-1] = cont[1:, 0:2]
        aux_target[-1] = target 
        aux_target_tensor = torch.tensor(aux_target, dtype=torch.float32)
        
        # Return index for safe inference
        return img, cont_tensor, cat_tensor, target_tensor, aux_target_tensor, target_polar_tensor, idx


In [4]:
def multimodal_collate_fn(batch):
    # [FIXED] Handle index return
    imgs, conts, cats, targets, aux_targets, polar_targets, indices = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    targets = torch.stack(targets, dim=0)
    aux_targets_padded = pad_sequence(aux_targets, batch_first=True)
    polar_targets_batched = torch.stack(polar_targets, dim=0)
    indices = torch.tensor(indices, dtype=torch.long)
    return imgs_batched, conts_padded, cats_padded, lengths, targets, aux_targets_padded, polar_targets_batched, indices


In [5]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv(x_cat)
        return self.sigmoid(x_out)

class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.sa = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        x = self.features(x)
        sa_map = self.sa(x)
        x = x * sa_map
        x = self.pool(x).flatten(1)
        x = F.relu(self.fc(x))
        return x

class LSTMAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(LSTMAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, rnn_output):
        attn_weights = torch.softmax(self.attention(rnn_output), dim=1)
        return torch.sum(attn_weights * rnn_output, dim=1)

class PolarMultiModalNet(nn.Module):
    """
    Polar Coordinate Regression Model
    Output: (distance, sin_theta, cos_theta)
    """
    def __init__(self, input_dim_cont, num_types, num_results, gru_hidden=96):
        super(PolarMultiModalNet, self).__init__()
        self.cnn = ImprovedCNN()
        self.type_emb = nn.Embedding(num_types, 8)
        self.result_emb = nn.Embedding(num_results, 8)
        
        total_input_dim = input_dim_cont + 8 + 8
        self.gru_hidden = gru_hidden
        
        # Split GRU (Anti-Leakage)
        self.gru_fwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=False,
            dropout=0.1
        )
        
        self.gru_bwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=False,
            dropout=0.1
        )
        
        self.gru_attn = LSTMAttention(gru_hidden * 2)
        self.gru_fc = nn.Linear(gru_hidden * 2, 128)
        
        # [Polar] Distance Head
        self.distance_head = nn.Sequential(
            nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus()  # Distance is always positive
        )
        
        # [Polar] Direction Head (sin, cos)
        self.direction_head = nn.Sequential(
            nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2),
            nn.Tanh()  # sin, cos are in [-1, 1]
        )
        
        # Auxiliary head (for deep supervision, uses forward GRU only)
        self.aux_fc = nn.Linear(gru_hidden, 2)

    def forward(self, img, cont, cat, lengths):
        # 1. Feature Extraction
        img_feat = self.cnn(img)
        emb_type = self.type_emb(cat[:, :, 0])
        emb_result = self.result_emb(cat[:, :, 1])
        x_seq = torch.cat([cont, emb_type, emb_result], dim=2)
        
        # 2. Forward GRU
        packed_fwd = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_fwd_packed, _ = self.gru_fwd(packed_fwd)
        out_fwd, _ = pad_packed_sequence(out_fwd_packed, batch_first=True)
        
        # 3. Backward GRU (Manual Flip)
        x_seq_bwd = x_seq.clone()
        for i, length in enumerate(lengths):
            x_seq_bwd[i, :length] = x_seq[i, :length].flip(0)
            
        packed_bwd = pack_padded_sequence(x_seq_bwd, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_bwd_packed, _ = self.gru_bwd(packed_bwd)
        out_bwd, _ = pad_packed_sequence(out_bwd_packed, batch_first=True)
        
        # Reverse output back to original order
        for i, length in enumerate(lengths):
            out_bwd[i, :length] = out_bwd[i, :length].flip(0)
            
        # 4. Concatenate: [Forward, Backward]
        gru_out_combined = torch.cat([out_fwd, out_bwd], dim=2)  # [B, L, H*2]
        
        # 5. Main Output (Polar)
        gru_ctx = self.gru_attn(gru_out_combined)
        seq_feat = F.relu(self.gru_fc(gru_ctx))
        concat_feat = torch.cat([img_feat, seq_feat], dim=1)  # [B, 256]
        
        # Predict polar coordinates
        distance = self.distance_head(concat_feat).squeeze(-1)  # [B]
        direction = self.direction_head(concat_feat)  # [B, 2]
        sin_theta = direction[:, 0]
        cos_theta = direction[:, 1]
        
        # 6. Aux Output (Deep Supervision, Cartesian)
        aux_out = self.aux_fc(out_fwd)  # [B, L, 2]
        
        return distance, sin_theta, cos_theta, aux_out

# [Polar] Loss Functions
class PolarLoss(nn.Module):
    def __init__(self, w_distance=1.0, w_direction=2.0, huber_delta=1.0):
        super(PolarLoss, self).__init__()
        self.w_distance = w_distance
        self.w_direction = w_direction
        self.huber = nn.HuberLoss(delta=huber_delta)
    
    def forward(self, pred_d, pred_sin, pred_cos, target_polar):
        """
        Args:
            pred_d: (B,) predicted distance
            pred_sin: (B,) predicted sin(theta)
            pred_cos: (B,) predicted cos(theta)
            target_polar: (B, 3) [d, sin_theta, cos_theta]
        """
        target_d = target_polar[:, 0]
        target_sin = target_polar[:, 1]
        target_cos = target_polar[:, 2]
        
        # Distance Loss (Huber)
        loss_distance = self.huber(pred_d, target_d)
        
        # Direction Loss (Cosine Similarity)
        # 중요: 예측 벡터를 단위 벡터로 정규화해야 각도 오차만 정확히 계산됨
        pred_vec_norm = torch.sqrt(pred_sin**2 + pred_cos**2 + 1e-8)
        pred_sin = pred_sin / pred_vec_norm
        pred_cos = pred_cos / pred_vec_norm
        
        # cos(θ_pred - θ_true) = cos_pred*cos_true + sin_pred*sin_true
        cos_diff = pred_cos * target_cos + pred_sin * target_sin
        loss_direction = 1 - cos_diff.mean()  # 1 - cos(θ) → 0 when aligned
        
        # Total Loss
        total_loss = self.w_distance * loss_distance + self.w_direction * loss_direction
        
        return total_loss
        
class MaskedSeqEuclideanLoss(nn.Module):
    def __init__(self):
        super(MaskedSeqEuclideanLoss, self).__init__()
        
    def forward(self, pred, target, lengths):
        mask = torch.arange(pred.size(1), device=pred.device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1)  # [B, L, 1]
        
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        
        diff = pred_real - target_real
        dist = torch.sqrt(torch.sum(diff**2, dim=2) + 1e-6)  # [B, L]
        
        # Masking
        dist = dist * mask.squeeze(-1)
        
        return dist.sum() / mask.sum()


## 4. 학습

In [6]:
# [Polar] Training Function (Fixed with Distance-Aware Loss & Score Logging)
def train_multimodal_polar(train_df, n_splits=10, epochs=100, batch_size=64, lr=0.001):
    import sys
    sys.path.append('src')
    from polar_utils import normalize_vec, polar_to_cartesian
    from src.polar_loss import PolarLoss  # [New] Improved Loss
    
    gkf = GroupKFold(n_splits=n_splits)
    groups = train_df['game_id'] 
    
    fold_scores = []
    input_dim_cont = None
    num_types = None
    num_results = None
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
        fold_seed = FOLD_SEEDS.get(fold + 1, 42)
        seed_everything(fold_seed)
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        
        train_sub_df = train_df.iloc[train_idx].copy()
        val_sub_df = train_df.iloc[val_idx].copy()
        
        preprocessor = FootballPreprocessorPolar()
        preprocessor.fit(train_sub_df)

        if input_dim_cont is None:
            input_dim_cont = preprocessor.get_input_dim()
            num_types, num_results = preprocessor.get_num_classes()
            print(f"Model Dimensions: input={input_dim_cont}, types={num_types}, results={num_results}")
        
        train_episodes = preprocessor.transform(train_sub_df, is_train=True)
        val_episodes = preprocessor.transform(val_sub_df, is_train=True)
        
        train_dataset = MultiModalDataset(train_episodes, augment=True, cache_images=True)
        val_dataset = MultiModalDataset(val_episodes, augment=False, cache_images=True)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=multimodal_collate_fn, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=multimodal_collate_fn)
        
        model = PolarMultiModalNet(input_dim_cont, num_types, num_results, gru_hidden=96).to(DEVICE)
        
        # [New] Distance-Aware Weighted Loss
        criterion_main = PolarLoss(
            w_distance=2.0,   # Emphasis on Distance
            w_direction=1.0,  # Lower emphasis on Direction
            w_euclidean=1.0,  # [New] Direct Euclidean Optimization
            huber_delta=1.0
        )
        criterion_aux = MaskedSeqEuclideanLoss() 
        
        optimizer = optim.Adam(model.parameters(), lr=lr) 
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_dist = float('inf')
        patience_counter = 0
        patience_limit = 15
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            
            # Decay Aux Weight
            if epoch < 5:
                aux_weight = 0.5
            elif epoch < 20:
                aux_weight = 0.2
            else:
                aux_weight = 0.05

            for batch in train_loader:
                imgs, cont, cat, lengths, target, aux_target, polar_target, _ = batch
                imgs = imgs.to(DEVICE)
                cont = cont.to(DEVICE)
                cat = cat.to(DEVICE)
                lengths = lengths.to(DEVICE)
                aux_target = aux_target.to(DEVICE)
                polar_target = polar_target.to(DEVICE)
                
                optimizer.zero_grad()
                pred_d, pred_sin, pred_cos, aux_pred = model(imgs, cont, cat, lengths)
                
                loss_m = criterion_main(pred_d, pred_sin, pred_cos, polar_target)
                loss_a = criterion_aux(aux_pred, aux_target, lengths)
                
                loss = loss_m + aux_weight * loss_a
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            # Validation
            model.eval()
            val_dists = []
            with torch.no_grad():
                for batch_idx, batch in enumerate(val_loader):
                    imgs, cont, cat, lengths, target, aux_target, _, _ = batch  # Ignore indices
                    imgs = imgs.to(DEVICE)
                    cont = cont.to(DEVICE)
                    cat = cat.to(DEVICE)
                    lengths = lengths.to(DEVICE)
                    
                    pred_d_norm, pred_sin, pred_cos, _ = model(imgs, cont, cat, lengths)
                    
                    pred_d_norm = pred_d_norm.cpu().numpy()
                    pred_sin = pred_sin.cpu().numpy()
                    pred_cos = pred_cos.cpu().numpy()
                    
                    batch_start = batch_idx * batch_size
                    
                    for i in range(len(pred_d_norm)):
                        if batch_start + i >= len(val_dataset.episodes):
                            break
                        
                        ep = val_dataset.episodes[batch_start + i]
                        
                        pred_end_x, pred_end_y = polar_to_cartesian(
                            pred_d_norm[i], pred_sin[i], pred_cos[i],
                            ep['start_x'], ep['start_y'],
                            d_scale=ep['d_scale'],
                            normalize_direction=True,
                            clip_field=True
                        )
                        
                        true_end = ep['target_raw']
                        dist = np.sqrt((pred_end_x - true_end[0])**2 + (pred_end_y - true_end[1])**2)
                        val_dists.append(dist)
            
            mean_dist = np.mean(val_dists)
            scheduler.step(mean_dist)
            print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Dist {mean_dist:.4f}")
            
            if mean_dist < best_dist:
                best_dist = mean_dist
                torch.save(model.state_dict(), f"models/multimodal_polar_h96_{fold+1}.pth")
                joblib.dump(preprocessor, f"models/preprocessor_polar_h96_{fold+1}.pkl")
                print(f"  -> Saved Best Model (Dist: {best_dist:.4f})")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    print("Early Stopping")
                    break
        
        print(f"Fold {fold+1} Finished. Best Valid Dist: {best_dist:.4f}")
        fold_scores.append(best_dist)
    
    # [New] Improved Score Logging
    print("\n" + "="*40)
    print(f"{'FINAL FOLD SCORES':^40}")
    print("="*40)
    for i, score in enumerate(fold_scores):
        print(f"Fold {i+1:02d}: {score:.4f}")
    print("-" * 40)
    print(f"AVERAGE: {np.mean(fold_scores):.4f}")
    print("="*40 + "\n")
    
    return fold_scores



In [7]:
# [Polar] Train Model
train_multimodal_polar(train_df, n_splits=10, epochs=100, batch_size=64, lr=0.001)



=== Fold 1/10 ===
Model Dimensions: input=10, types=27, results=21
Pre-rendering 13847 images (Cache Enabled)...


Caching Images: 100%|██████████| 13847/13847 [00:01<00:00, 13233.59it/s]


Pre-rendering 1581 images (Cache Enabled)...


Caching Images: 100%|██████████| 1581/1581 [00:00<00:00, 13599.08it/s]


Epoch 1: Train Loss 36.3981, Val Dist 17.4347
  -> Saved Best Model (Dist: 17.4347)
Epoch 2: Train Loss 29.7128, Val Dist 16.7180
  -> Saved Best Model (Dist: 16.7180)
Epoch 3: Train Loss 28.4062, Val Dist 17.0221
Epoch 4: Train Loss 27.7392, Val Dist 16.0545
  -> Saved Best Model (Dist: 16.0545)
Epoch 5: Train Loss 27.2825, Val Dist 16.2889
Epoch 6: Train Loss 11.1261, Val Dist 16.1451
Epoch 7: Train Loss 11.0407, Val Dist 15.8128
  -> Saved Best Model (Dist: 15.8128)
Epoch 8: Train Loss 10.9798, Val Dist 15.8104
  -> Saved Best Model (Dist: 15.8104)
Epoch 9: Train Loss 10.9234, Val Dist 15.5449
  -> Saved Best Model (Dist: 15.5449)
Epoch 10: Train Loss 10.8670, Val Dist 15.3598
  -> Saved Best Model (Dist: 15.3598)
Epoch 11: Train Loss 10.8352, Val Dist 15.1820
  -> Saved Best Model (Dist: 15.1820)
Epoch 12: Train Loss 10.7839, Val Dist 15.1330
  -> Saved Best Model (Dist: 15.1330)
Epoch 13: Train Loss 10.7253, Val Dist 15.1149
  -> Saved Best Model (Dist: 15.1149)
Epoch 14: Train Lo

...

Caching Images: 100%|██████████| 13858/13858 [00:00<00:00, 14306.12it/s]


Pre-rendering 1570 images (Cache Enabled)...


Caching Images: 100%|██████████| 1570/1570 [00:00<00:00, 13764.69it/s]


Epoch 1: Train Loss 36.5008, Val Dist 17.9561
  -> Saved Best Model (Dist: 17.9561)
Epoch 2: Train Loss 29.7039, Val Dist 16.9218
  -> Saved Best Model (Dist: 16.9218)
Epoch 3: Train Loss 28.4259, Val Dist 16.4428
  -> Saved Best Model (Dist: 16.4428)
Epoch 4: Train Loss 27.7940, Val Dist 16.7781
Epoch 5: Train Loss 27.3378, Val Dist 16.5967
Epoch 6: Train Loss 11.1408, Val Dist 16.1125
  -> Saved Best Model (Dist: 16.1125)
Epoch 7: Train Loss 11.0693, Val Dist 16.0684
  -> Saved Best Model (Dist: 16.0684)
Epoch 8: Train Loss 11.0003, Val Dist 15.8142
  -> Saved Best Model (Dist: 15.8142)
Epoch 9: Train Loss 10.9446, Val Dist 15.9056
Epoch 10: Train Loss 10.9057, Val Dist 16.2241
Epoch 11: Train Loss 10.8478, Val Dist 15.4572
  -> Saved Best Model (Dist: 15.4572)
Epoch 12: Train Loss 10.7975, Val Dist 15.3758
  -> Saved Best Model (Dist: 15.3758)
Epoch 13: Train Loss 10.7586, Val Dist 15.0524
  -> Saved Best Model (Dist: 15.0524)
Epoch 14: Train Loss 10.7501, Val Dist 15.3505
Epoch 15:

[np.float64(13.653775840786698),
 np.float64(13.530112266803945),
 np.float64(13.585134064930466),
 np.float64(13.763825114665499),
 np.float64(13.568061811524826),
 np.float64(13.724868084352737),
 np.float64(13.665983011036777),
 np.float64(13.609942018543851),
 np.float64(14.213962780144627),
 np.float64(13.726858415210453)]

## 5. Inference (Polar → Cartesian)

## 6. Create Submission

## 5. Inference (Polar Model with Rigorous TTA)

In [7]:
# ============================================================
# [FIXED] Robust Test Loading & Inference Pipeline (Polar, 10-Fold, TTA)
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import joblib
from tqdm import tqdm
from torch.utils.data import DataLoader
import sys
sys.path.append("src")
from polar_utils import polar_to_cartesian

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
N_FOLDS = 10
USE_TTA = True

SUBMISSION_PATH = "sample_submission.csv"
TEST_META_PATH = "test.csv"
MODEL_DIR = "models"
OUT_PATH = "submission_polar_h96_optimized.csv"

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def _fix_rel_path(p: str) -> str:
    return p[2:] if isinstance(p, str) and p.startswith("./") else p

def load_full_test_df(test_meta_path: str) -> pd.DataFrame:
    """Merge all episode CSVs referenced in test.csv into one DataFrame."""
    test_meta = pd.read_csv(test_meta_path)
    all_rows = []

    for _, row in tqdm(test_meta.iterrows(), total=len(test_meta), desc="Reading episode CSVs"):
        path = _fix_rel_path(row["path"])
        if not os.path.exists(path):
            pass
        df = pd.read_csv(path)
        df = df.reset_index(drop=True)
        df["row_idx"] = np.arange(len(df), dtype=np.int32)
        df["game_id"] = row["game_id"]
        # In test.csv, game_episode is the unique string ID (e.g., '12345_1')
        df["game_episode"] = row["game_episode"]
        all_rows.append(df)

    full_df = pd.concat(all_rows, ignore_index=True)
    full_df = full_df.sort_values(["game_id", "game_episode", "row_idx"]).reset_index(drop=True)
    return full_df

def sanity_check_alignment(test_df_full: pd.DataFrame, submission: pd.DataFrame):
    uniq = test_df_full[["game_id", "game_episode"]].drop_duplicates()
    if len(uniq) != len(submission):
        print(f"⚠️ WARNING: unique episodes ({len(uniq)}) != submission rows ({len(submission)})")
    else:
        print("✅ Episode count matches submission.")

def build_canonical_test_episodes(base_preprocessor, test_df_full: pd.DataFrame):
    """Create test_episodes ONCE to enforce a single canonical order."""
    test_episodes = base_preprocessor.transform(test_df_full, is_train=False)
    keys = []
    for ep in test_episodes:
        gid = ep.get("game_id", 0) 
        gep = ep.get("game_episode", 0)
        keys.append((gid, gep))
    return test_episodes, keys

def infer_fold_preds(model, preprocessor, test_episodes, tta: bool):
    """Return [N, 3] predictions (d_norm, sin, cos) for a single fold."""
    test_dataset = MultiModalDataset(test_episodes, augment=False, cache_images=True)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=multimodal_collate_fn
    )

    preds = []
    model.eval()

    if tta:
        scaler = preprocessor.scaler
        mean = torch.tensor(scaler.mean_, device=DEVICE, dtype=torch.float32)
        std = torch.tensor(scaler.scale_, device=DEVICE, dtype=torch.float32)

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Infer"):
            imgs, cont, cat, lengths, _, _, _, _ = batch

            imgs = imgs.to(DEVICE)
            cont = cont.to(DEVICE)
            cat = cat.to(DEVICE)
            lengths = lengths.to(DEVICE)

            p_d, p_sin, p_cos, _ = model(imgs, cont, cat, lengths)

            if tta:
                imgs_f = torch.flip(imgs, dims=[2])
                cont_raw = cont * std + mean
                if cont_raw.shape[-1] > 5:
                    cont_raw[:, :, 1] = 1.0 - cont_raw[:, :, 1]
                    cont_raw[:, :, 3] = 1.0 - cont_raw[:, :, 3]
                    cont_raw[:, :, 5] = -cont_raw[:, :, 5]
                cont_f = (cont_raw - mean) / std

                p_d_f, p_sin_f, p_cos_f, _ = model(imgs_f, cont_f, cat, lengths)
                p_sin_f = -p_sin_f
                p_d = 0.5 * (p_d + p_d_f)
                p_sin = 0.5 * (p_sin + p_sin_f)
                p_cos = 0.5 * (p_cos + p_cos_f)

            p_d = p_d.detach().cpu().numpy().reshape(-1)
            p_sin = p_sin.detach().cpu().numpy().reshape(-1)
            p_cos = p_cos.detach().cpu().numpy().reshape(-1)

            pred = np.stack([p_d, p_sin, p_cos], axis=1)
            preds.append(pred)

    preds = np.concatenate(preds, axis=0)
    return preds

def polar_preds_to_cartesian(test_episodes, avg_preds):
    final_xy = []
    pred_keys = []
    for i, ep in enumerate(test_episodes):
        d_norm = float(avg_preds[i, 0])
        sin_t = float(avg_preds[i, 1])
        cos_t = float(avg_preds[i, 2])

        p_x, p_y = polar_to_cartesian(
            d_norm, sin_t, cos_t,
            ep["start_x"], ep["start_y"],
            d_scale=ep["d_scale"],
            normalize_direction=True,
            clip_field=True
        )
        final_xy.append([p_x, p_y])
        pred_keys.append((ep.get("game_id", None), ep.get("game_episode", None)))
    return np.array(final_xy), pred_keys

# ------------------------------------------------------------
# Main Execution Block
# ------------------------------------------------------------
if __name__ == "__main__":
    if not os.path.exists(SUBMISSION_PATH) or not os.path.exists(TEST_META_PATH):
        print(f"⚠️ Files not found! {SUBMISSION_PATH} or {TEST_META_PATH}")
    else:
        submission = pd.read_csv(SUBMISSION_PATH)
        test_df_full = load_full_test_df(TEST_META_PATH)
        sanity_check_alignment(test_df_full, submission)

        # 1. Load Base Preprocessor
        base_preprocessor_path = os.path.join(MODEL_DIR, "preprocessor_polar_h96_1.pkl")
        if not os.path.exists(base_preprocessor_path):
            print(f"⚠️ Base preprocessor not found! {base_preprocessor_path}")
        else:
            base_preprocessor = joblib.load(base_preprocessor_path)
            test_episodes, canonical_keys = build_canonical_test_episodes(base_preprocessor, test_df_full)
            
            # 2. K-Fold Inference
            all_fold_preds = []
            for fold in range(1, N_FOLDS + 1):
                print(f"\n=== Inference Fold {fold}/{N_FOLDS} ===")
                
                pp_path = os.path.join(MODEL_DIR, f"preprocessor_polar_h96_{fold}.pkl")
                md_path = os.path.join(MODEL_DIR, f"multimodal_polar_h96_{fold}.pth")
                
                if not os.path.exists(pp_path) or not os.path.exists(md_path):
                    continue
                
                preprocessor = joblib.load(pp_path)
                
                # [ROBUST FIX 1] Dynamic Model Init from Checkpoint
                state_dict = torch.load(md_path, map_location=DEVICE)
                chk_types = state_dict['type_emb.weight'].shape[0]
                chk_results = state_dict['result_emb.weight'].shape[0]
                input_dim_cont = preprocessor.get_input_dim()
                
                model = PolarMultiModalNet(
                    input_dim_cont, 
                    chk_types, 
                    chk_results, 
                    gru_hidden=96
                ).to(DEVICE)
                
                model.load_state_dict(state_dict, strict=True)
                
                fold_preds = infer_fold_preds(model, preprocessor, test_episodes, tta=USE_TTA)
                all_fold_preds.append(fold_preds)
            
            if len(all_fold_preds) > 0:
                all_fold_preds = np.stack(all_fold_preds, axis=0) # [F, N, 3]
                avg_preds = all_fold_preds.mean(axis=0)           # [N, 3]
                
                coords, pred_keys = polar_preds_to_cartesian(test_episodes, avg_preds)
                
                # [ROBUST FIX 2] Merge Key Fix (Only use game_episode)
                pred_df = pd.DataFrame(coords, columns=["end_x", "end_y"])
                
                # pred_keys is [(game_id, game_episode), ...]
                # test.csv's game_episode maps to k[1] (e.g., '153363_1')
                pred_df["game_episode"] = [k[1] for k in pred_keys]
                
                # sample_submission only has 'game_episode', 'end_x', 'end_y' columns
                # We simply replace end_x/end_y by merging onto the template
                final_sub = submission[["game_episode"]].merge(
                    pred_df[["game_episode", "end_x", "end_y"]], 
                    on="game_episode", 
                    how="left"
                )
                
                final_sub["end_x"] = final_sub["end_x"].clip(0, 105)
                final_sub["end_y"] = final_sub["end_y"].clip(0, 68)
                final_sub = final_sub.fillna(0)
                
                final_sub.to_csv(OUT_PATH, index=False)
                print(f"✅ Submission saved: {OUT_PATH}")
                print(final_sub.head())
            else:
                print("❌ No folds inferred!")



Reading episode CSVs: 100%|██████████| 2414/2414 [00:03<00:00, 627.26it/s]


✅ Episode count matches submission.


C:\Users\semic\AppData\Local\Temp\ipykernel_49836\4044936417.py:179: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(md_path, map_location=DEVICE)



=== Inference Fold 1/10 ===
Pre-rendering 2414 images (Cache Enabled)...


Infer: 100%|██████████| 38/38 [00:02<00:00, 15.03it/s]


...


=== Inference Fold 9/10 ===
Pre-rendering 2414 images (Cache Enabled)...


Infer: 100%|██████████| 38/38 [00:02<00:00, 18.01it/s]



=== Inference Fold 10/10 ===
Pre-rendering 2414 images (Cache Enabled)...


Infer: 100%|██████████| 38/38 [00:02<00:00, 17.90it/s]

✅ Submission saved: submission_polar_h96_optimized.csv
  game_episode      end_x      end_y
0     153363_1  67.953806  12.721565
1     153363_2  30.406996  50.059748
2     153363_6  35.886074  61.653611
3     153363_7  51.559660   8.285633
4     153363_8  80.418792   9.127678


In [8]:
result_v7_foldseed_h96_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
result_v7_foldseed_h96_tta_corrected.head()

,game_episode,end_x,end_y
0,153363_1,63.744760,10.535255
1,153363_2,25.247570,50.701256
2,153363_6,26.461975,63.656200
3,153363_7,53.256310,4.663423
4,153363_8,81.452260,8.473086


In [9]:
result_v7_foldseed_h64_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v7_h64_tta_corrected.csv")
result_v7_foldseed_h64_tta_corrected.head()

,game_episode,end_x,end_y
0,153363_1,61.837930,9.095944
1,153363_2,25.033142,47.295547
2,153363_6,31.202053,63.940987
3,153363_7,53.329033,5.811836
4,153363_8,81.128800,8.247789


In [11]:
import os
import pandas as pd

# ✅ 너 파일명/경로만 맞춰서 수정
H96_PATH = "submission_multimodal/multimodal_v7_h96_tta_corrected.csv"   # H96 tta 결과
POLAR_PATH = "submission_polar_h96_optimized.csv"                       # polar 결과

OUT_DIR = "submission_ensemble"
os.makedirs(OUT_DIR, exist_ok=True)

h96 = pd.read_csv(H96_PATH)
polar = pd.read_csv(POLAR_PATH)

# game_episode 기준으로 정렬/매칭
df = h96.merge(polar, on="game_episode", suffixes=("_h96", "_polar"), how="inner")

assert len(df) == len(h96), "Merge mismatch: check game_episode uniqueness / file alignment"

def make_ens(w_h96: float):
    w_polar = 1.0 - w_h96
    out = pd.DataFrame({
        "game_episode": df["game_episode"],
        "end_x": (w_h96 * df["end_x_h96"] + w_polar * df["end_x_polar"]).clip(0, 105),
        "end_y": (w_h96 * df["end_y_h96"] + w_polar * df["end_y_polar"]).clip(0, 68),
    })
    return out

for w in [0.7, 0.6]:
    out = make_ens(w)
    out_path = f"{OUT_DIR}/ens_h96{int(w*100)}_polar{int((1-w)*100)}.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    print(out.head())


Saved: submission_ensemble/ens_h9670_polar30.csv
  game_episode      end_x      end_y
0     153363_1  65.007474  11.191148
1     153363_2  26.795398  50.508804
2     153363_6  29.289205  63.055423
3     153363_7  52.747315   5.750086
4     153363_8  81.142220   8.669463
Saved: submission_ensemble/ens_h9660_polar40.csv
  game_episode      end_x      end_y
0     153363_1  65.428378  11.409779
1     153363_2  27.311340  50.444653
2     153363_6  30.231615  62.855164
3     153363_7  52.577650   6.112307
4     153363_8  81.038873   8.734923


In [12]:
import pandas as pd
import numpy as np

FILES = {
    "H96": "submission_multimodal/multimodal_v7_h96_tta_corrected.csv",
    "H64": "submission_multimodal/multimodal_v7_h64_tta_corrected.csv",
    "Polar": "submission_polar_h96_optimized.csv",
    "Ens_70_30": "submission_ensemble/ens_h9670_polar30.csv",
    "Ens_60_40": "submission_ensemble/ens_h9660_polar40.csv",
}

def summarize(df, name):
    stats = {}
    for col in ["end_x", "end_y"]:
        s = df[col]
        stats[f"{col}_mean"] = s.mean()
        stats[f"{col}_std"] = s.std()
        stats[f"{col}_min"] = s.min()
        stats[f"{col}_max"] = s.max()
        stats[f"{col}_q25"] = s.quantile(0.25)
        stats[f"{col}_q75"] = s.quantile(0.75)
    return pd.Series(stats, name=name)

summary = []

for name, path in FILES.items():
    df = pd.read_csv(path)
    summary.append(summarize(df, name))

summary_df = pd.concat(summary, axis=1).T
pd.set_option("display.float_format", "{:.4f}".format)
print(summary_df)


           end_x_mean  end_x_std  end_x_min  end_x_max  end_x_q25  end_x_q75  \
H96           66.8816    21.7833     5.2596   101.4013    50.7169    85.8554   
H64           66.9383    21.4831     5.2903   100.4247    50.5651    85.8319   
Polar         68.2053    20.9974     0.0000   105.0000    53.7820    85.4752   
Ens_70_30     67.2787    21.4890     3.6817   101.5910    51.7970    85.6933   
Ens_60_40     67.4111    21.4018     3.1557   101.7682    52.0510    85.7804   

           end_y_mean  end_y_std  end_y_min  end_y_max  end_y_q25  end_y_q75  
H96           33.3093    22.7188     1.0082    66.9495     9.2238    57.7500  
H64           33.2392    22.5938     1.4247    66.3395     9.5039    57.5747  
Polar         33.4161    21.5765     0.0000    68.0000    12.2912    54.5446  
Ens_70_30     33.3413    22.3095     0.7058    66.8242    10.1520    56.8546  
Ens_60_40     33.3520    22.1853     0.6049    66.9534    10.4557    56.5071  


In [13]:
def pairwise_distance(df1, df2):
    dx = df1["end_x"] - df2["end_x"]
    dy = df1["end_y"] - df2["end_y"]
    dist = np.sqrt(dx**2 + dy**2)
    return {
        "mean_dist": dist.mean(),
        "std_dist": dist.std(),
        "max_dist": dist.max(),
    }

h96 = pd.read_csv(FILES["H96"])
polar = pd.read_csv(FILES["Polar"])

print("H96 vs Polar:", pairwise_distance(h96, polar))


H96 vs Polar: {'mean_dist': np.float64(4.483949349855328), 'std_dist': np.float64(3.122136108692705), 'max_dist': np.float64(25.112079929601844)}


In [14]:
import pandas as pd
import numpy as np

POLAR_PATH = "submission_polar_h96_optimized.csv"
df = pd.read_csv(POLAR_PATH)

N = len(df)

def pct(mask):
    return 100 * mask.sum() / N

stats = {
    # 정확히 경계
    "end_x == 0": pct(df["end_x"] == 0),
    "end_x == 105": pct(df["end_x"] == 105),
    "end_y == 0": pct(df["end_y"] == 0),
    "end_y == 68": pct(df["end_y"] == 68),

    # 거의 경계 (1m 이내)
    "end_x <= 1": pct(df["end_x"] <= 1),
    "end_x >= 104": pct(df["end_x"] >= 104),
    "end_y <= 1": pct(df["end_y"] <= 1),
    "end_y >= 67": pct(df["end_y"] >= 67),
}

print("=== Polar Boundary Hit Ratio (%) ===")
for k, v in stats.items():
    print(f"{k:15s}: {v:6.3f}%")


=== Polar Boundary Hit Ratio (%) ===
end_x == 0     :  0.041%
end_x == 105   :  0.041%
end_y == 0     :  0.994%
end_y == 68    :  0.704%
end_x <= 1     :  0.041%
end_x >= 104   :  0.041%
end_y <= 1     :  1.574%
end_y >= 67    :  1.408%


In [15]:
def boundary_stats(path, name):
    df = pd.read_csv(path)
    N = len(df)
    return {
        "model": name,
        "x_min_or_max_%": 100 * ((df["end_x"] <= 1) | (df["end_x"] >= 104)).sum() / N,
        "y_min_or_max_%": 100 * ((df["end_y"] <= 1) | (df["end_y"] >= 67)).sum() / N,
    }

rows = []
rows.append(boundary_stats("submission_multimodal/multimodal_v7_h96_tta_corrected.csv", "H96"))
rows.append(boundary_stats("submission_polar_h96_optimized.csv", "Polar"))

pd.DataFrame(rows)


,model,x_min_or_max_%,y_min_or_max_%
0,H96,0.0000,0.0000
1,Polar,0.0829,2.9826
